<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weekend 8 — AI Agents: AI That Acts
### Duration: 2 Hours

## Learning Objectives
- Explain, in plain English, the difference between a chatbot and an **agent**
- Walk through the agent loop — **perceive → think → act → observe → repeat** — step by step
- Read and write a **tool definition** (`name`, `description`, `input_schema`)
- Trace a real multi-step agent run turn by turn using `stop_reason`
- Identify the common ways agents fail and the exact fix for each

---

# Part 1 — From Chatbot to Agent

Before any code, let's get the core idea rock-solid, because everything this weekend builds on it.

## 🧪 Run-along setup (optional but recommended)

This is a *theory* session, but a few short snippets let you **see** each idea live. Add your key to Colab Secrets (🔑) as `MY_API_KEY`, then run the two cells below. Skip them if you just want to read.

In [1]:
# Install the SDK and create a client we reuse for the demo snippets
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 6.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata     # read Colab Secrets
import os, json
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")
import anthropic
client = anthropic.Anthropic()              # one client for all demos below
print("✅ Ready to run theory demos")

✅ Ready to run theory demos


## 🧠 A chatbot *talks*. An agent *acts*.

So far, every Claude call you've made has been a **conversation**: you send text, Claude sends text back. Useful — but Claude can only ever *describe* what to do. It can't actually do it.

An **agent** is Claude plus the ability to **take actions in the real world** — look something up, run a calculation, call an API, send an email — and then **react to the result**. The model stops being a narrator and becomes a doer.

### A concrete contrast
Ask each one: *"Is product SKU-2271 in stock at the Zepto Indiranagar store?"*

- **Chatbot:** "To check stock, you'd query your inventory system for SKU-2271 at that location."
- **Agent:** *calls* the inventory API → reads `units = 3` → replies "Yes — 3 units left, but that's below the reorder level, so I'd restock."

The chatbot gives you homework. The agent gives you the answer.

### 👀 See it: a chatbot can only *describe*
Run this — Claude has **no tools**, so it can only tell you *how* to check stock, not actually check it.

In [3]:
# A plain call with NO tools: Claude can talk, but cannot act
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[{"role": "user", "content": "Is SKU-2271 in stock at the Zepto Indiranagar store?"}]
)
print(response.content[0].text)
# ^ Notice it DESCRIBES the steps — it has no way to actually look anything up.

I don't have access to real-time inventory data or information about Zepto store locations and their stock levels. 

To check if SKU-2271 is in stock at the Zepto Indiranagar store, I'd recommend:

1. **Using the Zepto app or website** - Search for the product and check availability at your selected location
2. **Calling the store directly** - Contact Zepto Indiranagar store for immediate inventory confirmation
3. **Contacting Zepto customer service** - They can look up stock information for you

Is there anything else I can help you with?


## 🪜 Three analogies to lock it in

1. **Phone friend vs. office assistant.** A chatbot is a knowledgeable friend on the phone giving advice. An agent is an assistant *in your office* who can open the filing cabinet, make calls, and hand you a finished result.

2. **Recipe vs. cook.** A chatbot reads you the recipe. An agent walks into the kitchen, checks what's in the fridge, and cooks the dish — adjusting when an ingredient is missing.

3. **GPS voice vs. chauffeur.** A chatbot says "turn left in 400 metres." An agent *drives the car*, and re-routes when there's traffic.

## 🌍 Real agents in Indian companies

| Company | What the agent does (acts, not just talks) |
|---|---|
| **Razorpay** | Reads a failed-payment log, queries the gateway, drafts and applies the fix |
| **Swiggy** | Checks restaurant availability, ETA, and rider load before confirming an order |
| **Apollo Hospitals** | Finds an open slot, checks doctor availability, and books the appointment |
| **Zepto** | Watches stock levels, calculates reorders, and raises a purchase request |
| **Nykaa** | Pulls a customer's order history and processes a return end-to-end |

In every case the value is the **doing**, not the describing.

## ✅ Quick Check 1
> **Q:** A support bot replies: "You can reset your password from Settings → Security." Is this a chatbot or an agent?
> **A:** A **chatbot** — it only *describes* the steps. An agent would actually trigger the password-reset action for the user.

---

# Part 2 — The Agent Loop

Every agent, no matter how fancy, runs the same four-beat cycle until the goal is met.

## 🔁 The loop in plain English

```
        ┌─────────────────────────────────────────────┐
        │  1. PERCEIVE  — read the goal + latest result │
        │  2. THINK     — decide the next action        │
        │  3. ACT       — call a tool                   │
        │  4. OBSERVE   — read what the tool returned    │
        │        └──────────► repeat until done          │
        └─────────────────────────────────────────────┘
```

That's it. The "intelligence" is in step 2 (Claude deciding) and the "power" is in step 3 (actually doing something).

## 🔍 Each beat of the loop, explained

**1. Perceive** — the agent looks at everything it knows so far: the original goal, and the result of its last action. (On the API, this is the whole `messages` list.)

**2. Think** — Claude reasons: *do I have enough to answer, or do I need a tool? If a tool, which one, with what inputs?*

**3. Act** — Claude requests a specific tool call. **Your code runs it.** (Claude itself never executes anything.)

**4. Observe** — the tool's output is fed back to Claude so it can decide the *next* step — maybe another tool, maybe the final answer.

## 🧩 The same loop on the Claude API

The abstract loop maps exactly onto four concrete API moves:

1. You call `messages.create(...)` with a list of **tools** and the conversation so far.
2. Claude replies. If it wants to act, the reply has `stop_reason: "tool_use"` and one or more `tool_use` blocks (which tool, what arguments).
3. **Your code executes the tool** and appends the result as a `tool_result` message.
4. You call `messages.create(...)` again. Claude reads the result and either calls another tool or returns a final answer with `stop_reason: "end_turn"`.

## 🚦 `stop_reason` is the steering wheel

After every Claude reply, your code checks one field — `response.stop_reason` — to know what to do next:

| `stop_reason` | Meaning | Your code should... |
|---|---|---|
| `"tool_use"` | Claude wants to call a tool | Run the tool, feed the result back, loop again |
| `"end_turn"` | Claude is done, gave a final answer | Stop the loop, show the answer |
| `"max_tokens"` | Hit the `max_tokens` limit | Raise the limit or continue |

**The golden rule:** while `stop_reason == "tool_use"`, keep looping. Anything else → stop.

### 👀 See it: read the `stop_reason`
That plain answer above finished normally, so its `stop_reason` is `"end_turn"`. Your loop uses exactly this field to decide whether to keep going.

In [4]:
# The same response object from the snippet above
print("stop_reason:", response.stop_reason)   # expect: end_turn  (no tool was requested)

stop_reason: end_turn


> **Remember:** Claude **decides**, your code **executes**. Claude can no more run your `check_stock` function than a manager can personally rewire a server — it tells you what it needs; you do it and report back.

## ✅ Quick Check 2
> **Q:** Claude's reply has `stop_reason: "tool_use"`. What are the two things your code must do before calling Claude again?
> **A:** (1) Actually run the requested tool, and (2) append its output back into `messages` as a `tool_result`. Only then do you call Claude again.

---

# Part 3 — Tools: The Agent's Hands

An agent is only as capable as the tools you give it. Let's dissect a tool definition.

## 🔧 A tool has exactly three parts

```json
{
  "name": "check_stock",
  "description": "Get the current units in stock for a grocery item.",
  "input_schema": {
    "type": "object",
    "properties": {
      "item": { "type": "string", "description": "The item name, e.g. 'milk'" }
    },
    "required": ["item"]
  }
}
```

That JSON is *all* Claude knows about the tool. It never sees your Python — only this description.

## 1️⃣ `name` — the label your code dispatches on
A short identifier like `check_stock`. When Claude asks for a tool, it sends back this exact name, and **your code uses it to decide which function to call.** It must match on both sides.

## 2️⃣ `description` — the most important field

This is the plain-English explanation Claude reads to decide *whether and when* to use the tool. A vague description is the #1 cause of agents picking the wrong tool.

**Weak:** `"stock tool"`
**Strong:** `"Get the current units in stock for a single grocery item. Use when the user asks how much of something is available."`

Write descriptions like you're onboarding a new intern who can't ask follow-up questions.

## 3️⃣ `input_schema` — the arguments, typed

A JSON Schema describing what inputs the tool needs. It tells Claude the argument **names, types, and which are required**, so Claude fills them in correctly.

```json
"input_schema": {
  "type": "object",
  "properties": {
    "current": { "type": "integer", "description": "Units currently in stock" },
    "target":  { "type": "integer", "description": "Desired stock level" }
  },
  "required": ["current", "target"]
}
```

Tight schemas prevent Claude from inventing bad arguments.

### 👀 See it: a tool is just a Python dict
A tool definition is plain JSON. Build one and print it — this exact structure is everything Claude knows about the tool.

In [5]:
# A tool is a dictionary with three keys: name, description, input_schema
check_stock_tool = {
    "name": "check_stock",
    "description": "Get the current units in stock for a single grocery item.",
    "input_schema": {
        "type": "object",
        "properties": {"item": {"type": "string", "description": "The item name, e.g. 'milk'"}},
        "required": ["item"]
    }
}

# Print it as formatted JSON — this is literally what Claude reads
print(json.dumps(check_stock_tool, indent=2))

{
  "name": "check_stock",
  "description": "Get the current units in stock for a single grocery item.",
  "input_schema": {
    "type": "object",
    "properties": {
      "item": {
        "type": "string",
        "description": "The item name, e.g. 'milk'"
      }
    },
    "required": [
      "item"
    ]
  }
}


## 🤔 How does Claude pick a tool? (worked example)

You give Claude two tools: `check_stock` and `current_price`. The user asks: *"How much does bread cost?"*

- Claude reads both **descriptions**.
- `check_stock` says "units in stock" — doesn't match "cost".
- `current_price` says "unit price in rupees" — matches "cost".
- Claude returns a `tool_use` block for `current_price` with `{"item": "bread"}`.

The match happened purely on the **descriptions**. That's why they matter so much.

## ✅ Quick Check 3
> **Q:** Your agent keeps calling the wrong tool. Which part of the tool definition should you fix first?
> **A:** The **`description`**. Claude chooses tools by reading descriptions, so making them clearer and more specific fixes most wrong-tool problems.

---

# Part 4 — A Full Agent Run, Traced Turn by Turn

Let's watch the loop actually run. Goal: *"How many units of bread should we reorder to reach 20?"* The agent has two tools: `check_stock(item)` and `reorder_amount(current, target)`.

### Turn 1 — Claude needs the current stock
- **Perceive:** sees the goal. It can't reorder without knowing current stock.
- **Think:** "I need `check_stock` for bread first."
- **Act:** returns `tool_use` → `check_stock({"item": "bread"})`, `stop_reason="tool_use"`.
- **Your code:** runs it → bread = `4`. Appends `tool_result: "4"`.

### Turn 2 — Claude does the calculation
- **Perceive:** now knows bread = 4, target = 20.
- **Think:** "I have current and target — call `reorder_amount`."
- **Act:** returns `tool_use` → `reorder_amount({"current": 4, "target": 20})`.
- **Your code:** runs it → `16`. Appends `tool_result: "16"`.

### Turn 3 — Claude answers
- **Perceive:** has the reorder number, 16.
- **Think:** "I have everything. No more tools needed."
- **Act:** returns a final text answer: *"Reorder 16 units of bread to reach 20."*, `stop_reason="end_turn"`.
- **Your code:** sees `end_turn` → stops the loop and prints the answer.

### The whole trace at a glance
```
GOAL: reorder bread to reach 20
 ├─ Turn 1: check_stock(bread)        -> 4     [tool_use]
 ├─ Turn 2: reorder_amount(4, 20)     -> 16    [tool_use]
 └─ Turn 3: "Reorder 16 units."               [end_turn] ✅
```
Three turns, two tool calls, one answer. Notice Claude **chained** the tools: the output of turn 1 (4) became an input to turn 2.

### 👀 See it: the whole trace, live
Here's the entire run from above as a tiny, self-contained program. Run it and watch Claude **chain** the two tools — `check_stock` then `reorder_amount` — exactly as traced. (The lab breaks this down in detail; here just watch it happen.)

In [6]:
# --- tools (functions) ---
INVENTORY = {"milk": 12, "bread": 4, "eggs": 30}
def check_stock(item):      return INVENTORY.get(item.lower(), 0)
def reorder_amount(c, t):   return max(t - c, 0)

# --- tool definitions (what Claude sees) ---
tools = [
    {"name": "check_stock",
      "description": "Get current units in stock for a grocery item.",
      "input_schema": {"type": "object",
          "properties": {"item": {"type": "string"}}, "required": ["item"]}},
    {"name": "reorder_amount",
      "description": "Calculate units to reorder to reach a target stock level.",
      "input_schema": {"type": "object",
          "properties": {"current": {"type": "integer"}, "target": {"type": "integer"}},
          "required": ["current", "target"]}},
]

# --- dispatcher: our code runs whatever Claude asks for ---
def run_tool(name, args):
    if name == "check_stock":    return check_stock(args["item"])
    if name == "reorder_amount": return reorder_amount(args["current"], args["target"])

# --- the agent loop ---
messages = [{"role": "user",
    "content": "How many units of bread should we reorder to reach 20? Use the tools."}]
for turn in range(5):
    r = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=500, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": r.content})
    if r.stop_reason != "tool_use":
        print("\n🏁", r.content[0].text); break
    out = []
    for b in r.content:
        if b.type == "tool_use":
            res = run_tool(b.name, b.input)
            print(f"🔧 turn {turn}: {b.name}({b.input}) -> {res}")
            out.append({"type": "tool_result", "tool_use_id": b.id, "content": str(res)})
    messages.append({"role": "user", "content": out})

🔧 turn 0: check_stock({'item': 'bread'}) -> 4
🔧 turn 1: reorder_amount({'current': 4, 'target': 20}) -> 16

🏁 **Result:** You should reorder **16 units of bread** to reach your target stock level of 20 units (since you currently have 4 units in stock).


> **This chaining is the heart of agents.** No single tool could answer the question. The agent broke the goal into steps and combined the results — exactly what you'll build in the lab.

---

# Part 5 — Memory: How the Agent Remembers

An agent has no magic memory. It remembers by **keeping the whole conversation.**

## 🧠 The `messages` list *is* the memory

Every turn appends to one growing list: the user goal, each of Claude's replies, and every `tool_result`. On the next call you pass the *entire* list back, so Claude can "see" everything that happened.

```
messages = [
  user:      "reorder bread to reach 20",
  assistant: tool_use check_stock(bread),
  user:      tool_result "4",
  assistant: tool_use reorder_amount(4, 20),
  user:      tool_result "16",
  assistant: "Reorder 16 units."
]
```

That list is the agent's entire working memory for the task.

### 👀 See it: memory is just the growing list
After running the trace above, the `messages` list holds the entire run. Print each entry's role to see the agent's memory.

In [ ]:
# Walk the memory from the run above
print("The agent's memory has", len(messages), "messages:\n")
for i, m in enumerate(messages):
    if isinstance(m["content"], str):
        kind = "text goal"
    else:
        kind = ", ".join(getattr(b, "type", "text") if not isinstance(b, dict) else b.get("type", "?")
                         for b in m["content"])
    print(f"  [{i}] {m['role']:9s} -> {kind}")

## ⚠️ The classic memory bug
If you **forget to append the `tool_result`**, Claude calls the tool, but on the next turn it never sees the answer — so it calls the *same tool again*, forever. Symptom: an agent that loops on one tool and never finishes. Fix: always append the result back.

---

# Part 6 — Planning Patterns

How an agent organises its thinking. Same loop, different strategies for harder goals.

## 1. Step-by-step (reactive)
One action, look at the result, decide the next. Best for **simple, linear tasks.**

*Example:* "What's the cheapest item in stock?" → check price of each item one at a time, track the lowest. Each step depends only on the last.

## 2. Hierarchical (decompose)
Break a big goal into sub-goals, then tackle each. Best for **complex, multi-part tasks.**

*Example:* "Prepare the weekly restock report for Zepto Indiranagar." →
- Sub-goal A: list all items below reorder level.
- Sub-goal B: for each, compute reorder amount.
- Sub-goal C: total the cost.
- Sub-goal D: format the report.

Each sub-goal may use several tools.

## 3. Dynamic (re-plan)
Adjust the plan as new information arrives. Best for **unpredictable environments.**

*Example:* an agent booking an Apollo appointment finds the chosen doctor is on leave → it re-plans, searches for the next available doctor, and continues. The plan changed mid-task because reality did.

## 🧠 Extended thinking helps planning
For the harder patterns, you can turn on **extended thinking** (the `thinking` parameter with a token budget). It gives Claude a private scratchpad to reason about the plan *before* acting — fewer wrong turns on complex goals. *See docs.anthropic.com/en/docs/build-with-claude/extended-thinking.*

## ✅ Quick Check 4
> **Q:** "Book me any dentist appointment this week; if my usual clinic is full, try another." Which planning pattern is this?
> **A:** **Dynamic** — the agent must re-plan (switch clinics) when it discovers the first option is unavailable.

---

# Part 7 — Parallel Tool Use

When a goal needs several **independent** lookups, Claude can request **multiple tools in a single turn** instead of one at a time.

## ⚡ Example
Goal: *"Compare stock of milk, bread, and eggs."* These three lookups don't depend on each other, so in one turn Claude returns **three** `tool_use` blocks:

```
Turn 1:  check_stock(milk)   ┐
         check_stock(bread)  ├─ all in ONE reply
         check_stock(eggs)   ┘
Your code runs all three, returns all three results together.
Turn 2:  "Bread is lowest at 4 units."   [end_turn]
```

This is faster than three separate turns. Your loop handles it naturally if you iterate over *every* block in the reply (not just the first).

---

# Part 8 — Why Agents Fail (and the Exact Fix)

Most agent bugs fall into four buckets. Each has a tell-tale symptom and a precise fix.

## ❌ Failure 1 — Infinite loop
**Symptom:** the agent calls tools forever and never answers.
**Causes:** no iteration cap, or the `tool_result` isn't being fed back (so Claude never "sees" the answer).
**Fix:** cap the loop (e.g. max 5 turns) **and** always append `tool_result` to `messages`.

*Example:* an agent asked to check stock keeps calling `check_stock(bread)` 10 times — because the result was never appended.

## ❌ Failure 2 — Wrong tool
**Symptom:** the agent uses a tool that doesn't fit the question.
**Cause:** vague or overlapping `description` fields.
**Fix:** rewrite descriptions to be specific and non-overlapping.

*Example:* with descriptions "stock tool" and "price tool", Claude calls the wrong one for "how much is left?". Rewriting to "units in stock" vs "price in rupees" fixes it.

## ❌ Failure 3 — Hallucinated arguments
**Symptom:** the tool is called with made-up or wrongly-typed inputs.
**Cause:** a loose `input_schema` (missing types or `required`).
**Fix:** tighten the schema — declare types and mark required fields.

*Example:* Claude passes `{"item": "bread", "qty": "lots"}` because `qty` wasn't typed as an integer. Add `"qty": {"type": "integer"}`.

## ❌ Failure 4 — Ignores the result
**Symptom:** the agent gets a tool result but answers as if it didn't.
**Cause:** the `tool_result` block isn't linked to the request (`tool_use_id` mismatch) or wasn't appended.
**Fix:** return each result with the matching `tool_use_id`, appended to history.

## 🔍 How to diagnose any agent — print the trace

When something's wrong, **print every step**: which tool, what arguments, what came back, and the `stop_reason`. Almost every bug is obvious once you can *see* the loop:

```
🔧 turn 0: check_stock({'item': 'bread'}) -> 4      stop_reason=tool_use
🔧 turn 1: reorder_amount({'current': 4, 'target': 20}) -> 16   stop_reason=tool_use
🏁 turn 2: final answer                              stop_reason=end_turn
```

If you saw `check_stock` printed five times in a row, you'd instantly know the result wasn't being fed back.

## ✅ Quick Check 5
> **Q:** Your agent calls `check_stock` correctly but then answers "I don't have stock data." What's the likely bug?
> **A:** The `tool_result` isn't reaching Claude — either it wasn't appended to `messages`, or the `tool_use_id` doesn't match. Claude acted but never "observed" the result.

---

## 📖 Mini-glossary
- **Agent** — an LLM that can take actions via tools and react to results.
- **Tool** — a function you expose to Claude, defined by `name` + `description` + `input_schema`.
- **`tool_use`** — a `stop_reason` (and content block) meaning Claude wants to call a tool.
- **`tool_result`** — the message your code sends back with a tool's output.
- **`stop_reason`** — the field that tells your loop whether to continue or stop.
- **Agent loop** — perceive → think → act → observe → repeat.

## 🎯 Key Takeaways
- An **agent acts**, not just talks — it uses tools and reacts to results.
- The loop is **perceive → think → act → observe → repeat**, steered by `stop_reason`.
- Claude **decides** which tool; **your code executes** it and feeds the result back.
- A tool = `name` + `description` + `input_schema`; the **description drives tool choice**.
- The **`messages` list is the agent's memory** — always append `tool_result`.
- Agents shine by **chaining** tools (output of one feeds the next).
- Most failures: no stop condition, vague descriptions, loose schemas, or results not fed back.

---

## 🔍 Resources (Official)
- Tool use overview — https://docs.anthropic.com/en/docs/build-with-claude/tool-use
- Implement tool use — https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/implement-tool-use
- Extended thinking — https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking
- Messages API reference — https://docs.anthropic.com/en/api/messages

*Always verify the latest details at docs.anthropic.com.*